In [ ]:
%run ../globalvariables

In [ ]:
import mlflow
import pandas as pd
from datetime import date

In [ ]:
# Same widgets as forecast_gas
dbutils.widgets.dropdown("magnitud_target", "no2", [
    "no2", "no", "nox", "pm10", "pm2_5", "o3", "so2", "co",
    "tol", "ben", "ebe", "ch4", "nmhc", "tch",
])
dbutils.widgets.text("fecha", date.today().isoformat())

MAGNITUD_MAP = {
    "no2": "NO2", "no": "NO", "nox": "NOx", "pm10": "PM10", "pm2_5": "PM2.5",
    "o3": "O3", "so2": "SO2", "co": "CO", "tol": "TOL", "ben": "BEN",
    "ebe": "EBE", "ch4": "CH4", "nmhc": "NMHC", "tch": "TCH",
}
MAGNITUD_VALUE = MAGNITUD_MAP[dbutils.widgets.get("magnitud_target")]
FECHA = dbutils.widgets.get("fecha")

# Resolve that gas and date's experiment
mlflow.set_registry_uri("databricks-uc")
EXPERIMENT_PATH = f"{ML_EXPERIMENT_BASE}/{MAGNITUD_VALUE}_{FECHA}"
experiment = mlflow.get_experiment_by_name(EXPERIMENT_PATH)
print(EXPERIMENT_PATH)
print(experiment)

In [ ]:
# Every run as a DataFrame
runs = mlflow.search_runs(experiment_ids=[experiment.experiment_id], order_by=["start_time DESC"])
print(runs.shape)
display(runs)

In [ ]:
# Winner metrics per run
cols = ["start_time", "tags.mlflow.runName",
        "params.model_type", "metrics.mae", "metrics.rmse"]
display(runs[[c for c in cols if c in runs.columns]])

In [ ]:
# Forecast timeline last run
horizon = [f"metrics.pred_mean_h{h}" for h in range(1, FORECAST_HORIZON_GAS_DAYS + 1)]
last = runs[[c for c in horizon if c in runs.columns]].head(1)
display(last.T.rename(columns={last.index[0]: "valor_medio_predicho"}))